In [56]:
import pandas as pd
import requests
import re
import time
import os
from tqdm import tqdm

## EU

In [57]:
INPUT_DATASET_EU = './in_csv/eu/EU_companies_domains_validated.csv'
NEW_INPUT_DATASET_EU = './in_csv/eu/EU_domains_infos.csv'
EU_COMPANIES_INFO = 'Top-100-EU.csv'
OUTPUT_DOMAINS_CSV = './in_csv/eu/eu.csv'

# Load the dataset that contains domains and other technical data
data_eu = pd.read_csv(INPUT_DATASET_EU, index_col=0)

# Clean up the column names: lowercase, no spaces or dashes
data_eu.columns = (
    data_eu.columns.str.lower()
                   .str.replace(' ', '_')
                   .str.replace('-', '_')
)

# Load the file with additional company information
info_eu = pd.read_csv(EU_COMPANIES_INFO)

# Also clean the column names in this dataset
info_eu.columns = (
    info_eu.columns.str.lower()
                   .str.replace(' ', '_')
                   .str.replace('-', '_')
)

# Merge both datasets based on the company name without duplicates
data_eu = data_eu.merge(info_eu, how='left', left_on='company', right_on='company')

#remove duplicates with sufix_Y
data_eu = data_eu.loc[:,~data_eu.columns.duplicated()]


# Save the merged data back to the original file
data_eu.to_csv(NEW_INPUT_DATASET_EU, index=False)

# Now we only need the domain column for the next step
data_eu = data_eu[['domain']]

data_eu.to_csv(OUTPUT_DOMAINS_CSV, index=False)



In [58]:
import csv
import socket
import time
import select
import ssl
from OpenSSL import SSL, crypto
from cryptography import x509
from cryptography.hazmat.backends import default_backend
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import urlparse

# List to store failed domains
errored_domains = []

def fetch_ssl_certificate_chain(hostname, port=443, timeout=60, max_retries=3):
    for attempt in range(max_retries):
        try:
            # Ensure hostname does not contain scheme (https://)
            parsed_url = urlparse(hostname)
            if parsed_url.scheme:
                hostname = parsed_url.netloc
            
            # Create an SSL context with modern TLS support
            context = SSL.Context(SSL.TLS_METHOD)
            context.set_verify(SSL.VERIFY_NONE, lambda *x: True)
            
            # Create and connect socket
            sock = socket.create_connection((hostname, port), timeout=timeout)
            ssl_conn = SSL.Connection(context, sock)
            ssl_conn.set_connect_state()
            ssl_conn.set_tlsext_host_name(hostname.encode())
            
            end_time = time.time() + timeout
            while True:
                try:
                    ssl_conn.do_handshake()
                    break
                except (SSL.WantReadError, SSL.WantWriteError):
                    if time.time() > end_time:
                        raise TimeoutError("SSL handshake timed out")
                    select.select([ssl_conn], [ssl_conn], [], end_time - time.time())
                except SSL.Error as e:
                    if "tlsv1 alert protocol version" in str(e).lower():
                        print(f"[Attempt {attempt+1}] TLS version mismatch for {hostname}, trying TLSv1.2")
                        context = SSL.Context(SSL.TLSv1_2_METHOD)
                        continue
                    else:
                        raise e
            
            # Retrieve certificate chain
            cert_chain = ssl_conn.get_peer_cert_chain()
            cert_details = []
            for cert in cert_chain:
                pem_cert = crypto.dump_certificate(crypto.FILETYPE_PEM, cert).decode()
                parsed_cert = x509.load_pem_x509_certificate(pem_cert.encode(), default_backend())
                cert_details.append(extract_certificate_details(parsed_cert))
            
            # Close connection
            ssl_conn.shutdown()
            ssl_conn.close()
            sock.close()
            
            return cert_details
        
        except (socket.timeout, TimeoutError):
            print(f"[Attempt {attempt+1}] Timeout fetching certificate for {hostname}")
        except socket.gaierror:
            print(f"[Attempt {attempt+1}] Invalid hostname: {hostname} (Check if scheme is included)")
            break  # No need to retry
        except ConnectionRefusedError:
            print(f"[Attempt {attempt+1}] Connection refused for {hostname}")
            break  # No need to retry
        except SSL.Error as e:
            print(f"[Attempt {attempt+1}] SSL error for {hostname}: {e}")
        except Exception as e:
            print(f"[Attempt {attempt+1}] Unexpected error for {hostname}: {e}")
    
    # If all attempts failed, store the domain in errored_domains
    errored_domains.append(hostname)
    return []

def extract_certificate_details(cert):
    return {
        "subject": cert.subject.rfc4514_string(),
        "issuer": cert.issuer.rfc4514_string(),
        "serial_number": cert.serial_number,
        "not_before": cert.not_valid_before,
        "not_after": cert.not_valid_after,
    }

def write_cert_chain_to_csv(domain, cert_chain, filename):
    fieldnames = ["domain", "subject", "issuer", "serial_number", "not_before", "not_after"]
    with open(filename, mode='a', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        if file.tell() == 0:
            writer.writeheader()
        for cert in cert_chain:
            row = {"domain": domain}
            row.update(cert)
            writer.writerow(row)

def process_domain(domain, output_file):
    cert_chain = fetch_ssl_certificate_chain(domain)
    if cert_chain:
        write_cert_chain_to_csv(domain, cert_chain, output_file)

def process_domains_from_csv(input_file, output_file, error_log_file):
    with open(input_file, mode='r', newline='', encoding='utf-8') as infile:
        reader = csv.reader(infile)
        with ThreadPoolExecutor(max_workers=200) as executor:
            futures = {executor.submit(process_domain, row[0], output_file): row[0] for row in reader}
            for future in futures:
                try:
                    future.result()
                except Exception as e:
                    errored_domains.append(futures[future])
    
    # Log errored domains
    if errored_domains:
        with open(error_csv_file, 'w', newline='') as errfile:
            writer = csv.writer(errfile)
            writer.writerow(["Failed Domains"])
            for domain in errored_domains:
                writer.writerow([domain])



In [59]:
input_csv_file = './in_csv/eu/eu.csv'  # Replace with your input CSV file path
output_csv_file = './out_csv/eu/eu_certificates.csv'  # Replace with your output CSV file path
error_csv_file = './out_csv/eu/eu_failed_domains.csv'  # File to store failed domains
process_domains_from_csv(input_csv_file, output_csv_file, error_csv_file)

C:\Users\fsbertoglio\AppData\Local\Temp\ipykernel_30052\2169349664.py:87: CryptographyDeprecationWarning: Properties that return a naïve datetime object have been deprecated. Please switch to not_valid_before_utc.
  "not_before": cert.not_valid_before,
C:\Users\fsbertoglio\AppData\Local\Temp\ipykernel_30052\2169349664.py:88: CryptographyDeprecationWarning: Properties that return a naïve datetime object have been deprecated. Please switch to not_valid_after_utc.
  "not_after": cert.not_valid_after,


[Attempt 1] Invalid hostname: naturgy.com (Check if scheme is included)
[Attempt 1] Invalid hostname: terna.it (Check if scheme is included)


C:\Users\fsbertoglio\AppData\Local\Temp\ipykernel_30052\2169349664.py:55: CryptographyDeprecationWarning: Parsed a serial number which wasn't positive (i.e., it was negative or zero), which is disallowed by RFC 5280. Loading this certificate will cause an exception in a future release of cryptography.
  parsed_cert = x509.load_pem_x509_certificate(pem_cert.encode(), default_backend())
C:\Users\fsbertoglio\AppData\Local\Temp\ipykernel_30052\2169349664.py:86: CryptographyDeprecationWarning: Parsed a serial number which wasn't positive (i.e., it was negative or zero), which is disallowed by RFC 5280. Loading this certificate will cause an exception in a future release of cryptography.
  "serial_number": cert.serial_number,


[Attempt 1] Invalid hostname: buzziunicemusa.com (Check if scheme is included)
[Attempt 1] Invalid hostname: domain (Check if scheme is included)
[Attempt 1] SSL error for compagniedelodet.net: [('SSL routines', '', 'tlsv1 unrecognized name')]
[Attempt 1] Invalid hostname: schaeffler.com (Check if scheme is included)
[Attempt 1] SSL error for alphaholdings.gr: [('SSL routines', '', 'unsafe legacy renegotiation disabled')]
[Attempt 2] SSL error for compagniedelodet.net: [('SSL routines', '', 'tlsv1 unrecognized name')]
[Attempt 1] Connection refused for bmwusa.com
[Attempt 1] SSL error for thulegroup.com: [('SSL routines', '', 'tlsv1 alert internal error')]
[Attempt 1] Invalid hostname: gekterna.com (Check if scheme is included)
[Attempt 1] SSL error for inmocolonial.com: (10054, 'WSAECONNRESET')
[Attempt 3] SSL error for compagniedelodet.net: [('SSL routines', '', 'tlsv1 unrecognized name')]
[Attempt 2] SSL error for alphaholdings.gr: [('SSL routines', '', 'unsafe legacy renegotiation 

In [60]:
INPUT_CERTS_PATH = './out_csv/eu/eu_certificates.csv'
CLEAN_OUTPUT_PATH = './out_csv/eu/eu_clean_certificates.csv'

# Load the certificate data
data_eu = pd.read_csv(INPUT_CERTS_PATH, encoding='latin1')

# Remove duplicate rows based on the domain, keeping the first occurrence
data_eu = data_eu.drop_duplicates(subset='domain', keep='first')

# Function to parse a Distinguished Name (DN) string into a dictionary
def parse_dn(dn_string):
    dn_fields = {}
    pattern = r'(?<!\\),'  # split on commas that are not escaped
    parts = re.split(pattern, dn_string)
    for part in parts:
        if '=' in part:
            key, value = part.split('=', 1)
            dn_fields[key.strip()] = value.strip().replace('\\,', ',')
    return dn_fields

# Apply the parsing function to the 'subject' and 'issuer' columns
subject_parsed = data_eu['subject'].dropna().apply(parse_dn)
issuer_parsed = data_eu['issuer'].dropna().apply(parse_dn)

# Convert parsed data into DataFrames, adding prefixes to each field
subject_df = pd.json_normalize(subject_parsed).add_prefix('subject_')
issuer_df = pd.json_normalize(issuer_parsed).add_prefix('issuer_')

# Make sure the indexes match the original DataFrame
subject_df.index = subject_parsed.index
issuer_df.index = issuer_parsed.index

# Join the parsed subject and issuer fields back into the main DataFrame
data_eu = data_eu.join(subject_df).join(issuer_df)

# Define the standard DN fields we want to keep (excluding OIDs and unusual names)
dn_fields = ['CN', 'O', 'OU', 'L', 'ST', 'C']

# Create the full list of expected column names for subject and issuer
valid_subject_cols = [f'subject_{f}' for f in dn_fields]
valid_issuer_cols = [f'issuer_{f}' for f in dn_fields]

# Identify subject and issuer columns that are not in the standard list
subject_cols_to_drop = [col for col in data_eu.columns if col.startswith('subject_') and col not in valid_subject_cols]
issuer_cols_to_drop = [col for col in data_eu.columns if col.startswith('issuer_') and col not in valid_issuer_cols]

# Combine both lists into a single one for removal
cols_to_drop = subject_cols_to_drop + issuer_cols_to_drop

# Drop all non-standard subject and issuer columns
data_eu.drop(columns=cols_to_drop, inplace=True)

# Keep only the fields that are relevant for the analysis
data_eu = data_eu[['domain', 'subject_CN',
                   'issuer_CN', 'issuer_O', 'issuer_C',
                   'serial_number', 'not_before', 'not_after']]

# Save the cleaned dataset
data_eu.to_csv(CLEAN_OUTPUT_PATH, index=False)


In [61]:
CLEAN_CERTS_PATH = './out_csv/eu/eu_clean_certificates.csv'
COMPANIES_INFO_PATH = './in_csv/eu/EU_domains_infos.csv'
MERGED_OUTPUT_PATH = './out_csv/eu/merged_data.csv'

# Load cleaned certificate data and company information
extract_certificate_details = pd.read_csv(CLEAN_CERTS_PATH, encoding='latin1')
companies = pd.read_csv(COMPANIES_INFO_PATH, encoding='latin1')

# Merge the two datasets using the 'domain' column
merged_data = pd.merge(extract_certificate_details, companies, on='domain', how='left')

# Save the final merged dataset to a CSV file
merged_data.to_csv(MERGED_OUTPUT_PATH, index=False, sep=';', encoding='latin1')